# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nadim7009/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane 4, CTR opportunity scoring.** The model has one job: estimate the CTR a page should get, so that "below expectation" is measured against something fairer than a tier median.

- **Method:** gradient boosting for regression (`HistGradientBoostingRegressor`), trained with an absolute-error loss and weighted by impressions. Absolute error because CTR is skewed and a few pages with huge CTRs would otherwise drag a squared-error fit.
- **Features:** exact `avg_position`, `impressions_90d`, `content_type`, `main_intent`, `word_count`, `content_age_days`. These are page traits measured in the same 90-day window.
- **Target:** `ctr`, the observed CTR for each page (a percentage, so 0.24 means 0.24%). It is a measured outcome, not a label I made up with a rule.
- **Baseline to beat:** the week-4 rule, the median CTR of the page's position tier.

A simpler model (linear regression) would struggle with how position, intent and content type interact. A bigger one (a neural net) has no business on 12,000 rows. Boosting is the middle choice, and the baseline stays in the table so a reader can see whether it earned its place.

In [1]:
# Setup: works in Colab (clones my repo) and locally (walks up to the repo root)
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/nadim7009/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import json, pathlib
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import HistGradientBoostingRegressor
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
OUT = pathlib.Path("work/outputs"); OUT.mkdir(parents=True, exist_ok=True)
FIG = pathlib.Path("work/figures"); FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# Same candidate pool as weeks 1 to 4: real position data, first two result pages, >= 500 impressions
pool = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 500)].reset_index(drop=True)
print(f"{len(pool):,} pages from {pool['client_id'].nunique()} clients")

FEATS = ["avg_position", "impressions_90d", "content_type", "main_intent", "word_count", "content_age_days"]
Y = pool["ctr"].values
W = pool["impressions_90d"].values
GROUPS = pool["client_id"]

def make_X(cols):
    X = pool[cols].copy()
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            X[c] = X[c].fillna("unknown").astype("category")
    return X

def new_model():
    return HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05,
                                         categorical_features="from_dtype", random_state=42)

def wmae(y, p, w):
    return float(np.average(np.abs(y - p), weights=w))

def evaluate(cols, splitter, groups=None):
    """Return per-fold rule error, per-fold model error, and out-of-fold predictions for both."""
    X = make_X(cols)
    rule_err, model_err = [], []
    oof_rule = np.zeros(len(pool)); oof_model = np.zeros(len(pool))
    for tr, te in splitter.split(X, Y, groups):
        tier_median = pool.iloc[tr].groupby("position_tier")["ctr"].median()
        rp = pool.iloc[te]["position_tier"].map(tier_median).fillna(pool.iloc[tr]["ctr"].median()).values
        m = new_model().fit(X.iloc[tr], Y[tr], sample_weight=W[tr])
        mp = m.predict(X.iloc[te])
        oof_rule[te], oof_model[te] = rp, mp
        rule_err.append(wmae(Y[te], rp, W[te])); model_err.append(wmae(Y[te], mp, W[te]))
    return np.array(rule_err), np.array(model_err), oof_rule, oof_model

12,023 pages from 28 clients


## 2. Split design

**Grouped by client, 5 folds.** Each fold holds out whole clients. The model is then scored only on sites it never saw during training.

This is the honest split for this question. The editor will use the queue on sites where the model has no history of that site's own CTR habits. A random split would scatter one site's pages across train and test, and the model could learn "pages from this site look like this" and carry it into the test fold. Section 3 of the week-6 notebook measures how much that matters.

The metric is **impressions-weighted mean absolute error of CTR** (in CTR percentage points). Weighted, because a page seen 100,000 times matters more to an editor than one seen 500 times. Same pool, same metric as the week-2 framing. Seed 42 throughout.

In [2]:
rule_err, model_err, oof_rule, oof_model = evaluate(FEATS, GroupKFold(n_splits=5), GROUPS)

table = pd.DataFrame({"fold": range(1, 6), "tier_median_rule": rule_err.round(3), "boosting_model": model_err.round(3)})
table["model_better"] = model_err < rule_err   # unrounded, so a near tie is not mislabelled
table["gap_pts"] = (rule_err - model_err).round(4)
print(table.to_string(index=False))
rule_mean, model_mean = rule_err.mean(), model_err.mean()
print(f"\nMean weighted CTR error (held-out clients):  rule {rule_mean:.3f}   model {model_mean:.3f}"
      f"   model is {100 * (rule_mean - model_mean) / rule_mean:.1f}% lower")
print(f"Model better on {int(table['model_better'].sum())} of 5 folds")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(table["fold"], table["tier_median_rule"], "o-", label="tier-median rule", color="#888")
ax.plot(table["fold"], table["boosting_model"], "o-", label="boosting model", color="#1f6feb")
ax.set_xlabel("fold (each holds out different clients)"); ax.set_ylabel("weighted CTR error (pts)")
ax.set_xticks(range(1, 6)); ax.legend(); ax.set_title("Model vs rule on held-out clients")
fig.tight_layout(); fig.savefig(FIG / "fold_errors.png", dpi=150); plt.close(fig)

 fold  tier_median_rule  boosting_model  model_better  gap_pts
    1             0.190           0.196         False  -0.0056
    2             0.377           0.341          True   0.0366
    3             0.190           0.190          True   0.0004
    4             0.235           0.223          True   0.0111
    5             0.285           0.275          True   0.0098

Mean weighted CTR error (held-out clients):  rule 0.255   model 0.245   model is 4.1% lower
Model better on 4 of 5 folds


**How to read it.** The model's error is lower on average and in 4 of 5 folds, but by a small margin. In fold 3 the two are a near tie (see `gap_pts`), and in fold 1 the rule is slightly better. Fold 2 has the largest error for both methods, which suggests some clients are harder to predict from page traits alone. Treat this as directional: the model is slightly better than the lookup table on this slice, not clearly better.

Same OOF predictions, second question: does the model change *which pages* come out on top of the queue? A better error on average only matters if the top of the list moves.

In [3]:
def gap(expected, ctr=pool["ctr"].values, impr=W):
    return np.clip(expected - ctr, 0, None) / 100 * impr

pool["gap_rule"], pool["gap_model"] = gap(oof_rule), gap(oof_model)
top_rule = set(pool.sort_values("gap_rule", ascending=False).head(50).index)
top_model = set(pool.sort_values("gap_model", ascending=False).head(50).index)
overlap = len(top_rule & top_model)
zero_rule = int((pool.loc[list(top_rule), "clicks_90d"] == 0).sum())
zero_model = int((pool.loc[list(top_model), "clicks_90d"] == 0).sum())
print(f"Top-50 pages in both queues: {overlap} of 50")
print(f"Zero-click pages in the top 50:  rule {zero_rule}   model {zero_model}")

metrics = {"split": "GroupKFold(5) by client", "metric": "impressions-weighted MAE of ctr (points)",
           "pool_pages": int(len(pool)), "clients": int(pool["client_id"].nunique()),
           "rule_mae": round(float(rule_mean), 4), "model_mae": round(float(model_mean), 4),
           "rule_by_fold": rule_err.round(4).tolist(), "model_by_fold": model_err.round(4).tolist(),
           "model_better_folds": int(table["model_better"].sum()),
           "top50_overlap_rule_vs_model": overlap, "top50_zero_clicks_rule": zero_rule,
           "top50_zero_clicks_model": zero_model}
json.dump(metrics, open(OUT / "model_metrics.json", "w"), indent=2)
print("Wrote work/outputs/model_metrics.json")

Top-50 pages in both queues: 21 of 50
Zero-click pages in the top 50:  rule 1   model 1
Wrote work/outputs/model_metrics.json


## 4. Errors and interpretation

Where does the model miss, and what does it lean on? Two views: error by position tier, and permutation importance (how much worse the error gets when one feature is shuffled), measured on held-out clients only.

In [4]:
from sklearn.inspection import permutation_importance

by_tier = (pd.DataFrame({"tier": pool["position_tier"], "w": W,
                         "abs_err_model": np.abs(Y - oof_model), "abs_err_rule": np.abs(Y - oof_rule)})
           .groupby("tier").apply(lambda g: pd.Series({"pages": len(g),
                                                       "rule_wmae": np.average(g.abs_err_rule, weights=g.w),
                                                       "model_wmae": np.average(g.abs_err_model, weights=g.w)}),
                                  include_groups=False).round(3))
print("Weighted error by position tier (held-out clients):"); print(by_tier.to_string())

X = make_X(FEATS)
imp = np.zeros(len(FEATS))
for tr, te in GroupKFold(n_splits=5).split(X, Y, GROUPS):
    m = new_model().fit(X.iloc[tr], Y[tr], sample_weight=W[tr])
    r = permutation_importance(m, X.iloc[te], Y[te], sample_weight=W[te], n_repeats=3, random_state=42,
                               scoring="neg_mean_absolute_error")
    imp += r.importances_mean / 5
imp_s = pd.Series(imp, index=FEATS).sort_values(ascending=False)
print("\nPermutation importance (rise in weighted error when shuffled, held-out clients):")
print(imp_s.round(4).to_string())

Weighted error by position tier (held-out clients):
           pages  rule_wmae  model_wmae
tier                                   
page_1    7064.0      0.215       0.222
page_3_5    16.0      0.309       0.202
striking  4485.0      0.244       0.223
top_3      458.0      0.399       0.307

Permutation importance (rise in weighted error when shuffled, held-out clients):
avg_position        0.0113
content_age_days    0.0028
word_count          0.0021
main_intent         0.0002
content_type        0.0000
impressions_90d    -0.0030


**What this shows (observed, this slice only).**

- Most pages sit in the page-1 and striking-distance tiers. On page 1 the model is slightly *worse* than the rule (0.222 against 0.215); its gain comes from the striking tier and from `top_3`, which holds 458 pages and has a median CTR below page 1's, so I do not trust either method there. `page_3_5` has only 16 pages and is not worth reading.
- The model leans on `avg_position` first, then `content_age_days` and `word_count`. Intent and content type add almost nothing, and shuffling `impressions_90d` does not hurt (the small negative value is noise). This matches week 2: most of the CTR spread stays unexplained by the traits in this starter file.
- The two queues are different lists. Only 21 of the top 50 pages appear in both, so the choice between rule and model changes what an editor reads first. Neither list has a future outcome to test against yet, so I cannot say which one is right.
- The model is a small improvement, not a big win. Whether to keep it comes in week 6, after the split is audited.

## Self-check

- [x] Every section above is filled, with the thinking in markdown and the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime, then Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit the repo URL on the card.